<a href="https://colab.research.google.com/github/julianocmachado/uci-motion-rnn/blob/main/notebooks/01_exploracao/01_estrutura_e_carregamento.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fase 1 — Notebook 1: estrutura e carregamento dos dados

Neste notebook, vamos carregar somente os dados do **usuário 1** da base *Smartphone-Based Recognition of Human Activities and Postural Transitions*.

Ao final, teremos:

- `X`: matriz com as três acelerações e as três velocidades angulares;
- `y`: vetor com uma classe para cada amostra temporal;
- `id_experimento`: vetor que informa a origem de cada amostra;
- listas que mantêm os experimentos separados para evitar misturá-los posteriormente.

> Nesta etapa, ainda não faremos gráficos, pré-processamento ou treinamento de redes neurais.

## 1. Montagem do Google Drive

O comando abaixo permite que o Google Colab acesse os arquivos do seu Drive. A autorização ocorre entre o Colab e sua conta Google.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

## 2. Importação das bibliotecas

- `Path`: auxilia na construção dos caminhos das pastas e arquivos;
- `NumPy`: carrega e organiza os sinais em matrizes;
- `pandas`: organiza o arquivo de rótulos em uma tabela.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

## 3. Localização da base

Consideramos que a pasta `UCI-Motion-Database` está diretamente em `Meu Drive` e contém a pasta `RawData`.

In [ ]:
PASTA_BASE = Path('/content/drive/MyDrive/UCI-Motion-Database')
PASTA_RAW = PASTA_BASE / 'RawData'

if not PASTA_RAW.exists():
    raise FileNotFoundError(
        'A pasta RawData não foi encontrada. Verifique se a base foi '
        'descompactada dentro de UCI-Motion-Database.'
    )

arquivos_txt = sorted(PASTA_RAW.glob('*.txt'))

print('Pasta localizada:', PASTA_RAW)
print('Quantidade de arquivos .txt:', len(arquivos_txt))
print('Primeiros arquivos:')

for arquivo in arquivos_txt[:5]:
    print('-', arquivo.name)

O resultado esperado é uma pasta com **123 arquivos `.txt`**: 61 arquivos de acelerômetro, 61 de giroscópio e `labels.txt`.

## 4. Carregamento dos rótulos

Cada linha de `labels.txt` possui cinco valores:

1. número do experimento;
2. número do usuário;
3. classe da atividade;
4. amostra inicial do segmento;
5. amostra final do segmento.

In [ ]:
NOMES_COLUNAS = [
    'experimento',
    'usuario',
    'atividade',
    'inicio',
    'fim'
]

rotulos = pd.read_csv(
    PASTA_RAW / 'labels.txt',
    sep=r'\s+',
    names=NOMES_COLUNAS
)

print('Formato da tabela:', rotulos.shape)
rotulos.head()

## 5. Seleção do primeiro usuário

O usuário 1 participou dos experimentos 1 e 2. Em vez de escrever esses números manualmente, vamos identificá-los a partir da tabela de rótulos.

In [ ]:
USUARIO = 1

rotulos_usuario = rotulos.loc[
    rotulos['usuario'] == USUARIO
].copy()

experimentos = sorted(
    rotulos_usuario['experimento'].unique()
)

print('Usuário selecionado:', USUARIO)
print('Experimentos encontrados:', experimentos)
print('Quantidade de segmentos rotulados:', len(rotulos_usuario))

## 6. Função para carregar um experimento

Para cada experimento, carregaremos:

- acelerômetro: `acc_x`, `acc_y` e `acc_z`;
- giroscópio: `gyro_x`, `gyro_y` e `gyro_z`.

As duas matrizes serão combinadas horizontalmente, produzindo seis características por instante. O vetor `y` começará preenchido com zero. Depois, os intervalos descritos em `labels.txt` receberão as classes de 1 a 12. Assim, a classe 0 representa uma amostra sem rótulo explícito.

In [ ]:
def carregar_experimento(
    pasta_raw,
    numero_experimento,
    numero_usuario,
    tabela_rotulos
):
    nome_acc = (
        f'acc_exp{numero_experimento:02d}'
        f'_user{numero_usuario:02d}.txt'
    )
    nome_gyro = (
        f'gyro_exp{numero_experimento:02d}'
        f'_user{numero_usuario:02d}.txt'
    )

    caminho_acc = pasta_raw / nome_acc
    caminho_gyro = pasta_raw / nome_gyro

    if not caminho_acc.exists() or not caminho_gyro.exists():
        raise FileNotFoundError(
            f'Arquivos não encontrados para o experimento '
            f'{numero_experimento} e usuário {numero_usuario}.'
        )

    acelerometro = np.loadtxt(caminho_acc, dtype=np.float32)
    giroscopio = np.loadtxt(caminho_gyro, dtype=np.float32)

    if acelerometro.shape != giroscopio.shape:
        raise ValueError(
            'Acelerômetro e giroscópio possuem formatos diferentes.'
        )

    X_experimento = np.column_stack((acelerometro, giroscopio))
    y_experimento = np.zeros(len(X_experimento), dtype=np.int32)

    segmentos = tabela_rotulos.loc[
        (tabela_rotulos['experimento'] == numero_experimento)
        & (tabela_rotulos['usuario'] == numero_usuario)
    ]

    for segmento in segmentos.itertuples(index=False):
        inicio_python = int(segmento.inicio) - 1
        fim_python = int(segmento.fim)
        atividade = int(segmento.atividade)

        y_experimento[inicio_python:fim_python] = atividade

    return X_experimento, y_experimento

### Por que usamos `inicio - 1`, mas não `fim - 1`?

A base numera a primeira amostra como 1, enquanto o Python começa em 0. Por isso, subtraímos 1 do início. Em um *slice* do Python, a posição final não é incluída. Usar `fim` como limite faz com que a última amostra indicada pela base seja corretamente incluída.

## 7. Carregamento dos experimentos do usuário 1

Guardaremos cada experimento em uma posição diferente de uma lista. Isso será importante posteriormente, pois uma janela temporal não deve começar no experimento 1 e terminar no experimento 2.

In [ ]:
X_lista = []
y_lista = []
id_experimento_lista = []

for experimento in experimentos:
    X_exp, y_exp = carregar_experimento(
        pasta_raw=PASTA_RAW,
        numero_experimento=experimento,
        numero_usuario=USUARIO,
        tabela_rotulos=rotulos
    )

    X_lista.append(X_exp)
    y_lista.append(y_exp)
    id_experimento_lista.append(
        np.full(len(X_exp), experimento, dtype=np.int32)
    )

    print(
        f'Experimento {experimento:02d}: '
        f'X = {X_exp.shape} | y = {y_exp.shape}'
    )

Para os arquivos analisados, o resultado esperado é:

```text
Experimento 01: X = (20598, 6) | y = (20598,)
Experimento 02: X = (19286, 6) | y = (19286,)
```

## 8. Construção das matrizes finais

Vamos concatenar os dois experimentos para obter uma matriz geral do usuário 1. Os limites originais continuam disponíveis nas listas e no vetor `id_experimento`.

In [ ]:
X = np.concatenate(X_lista, axis=0)
y = np.concatenate(y_lista, axis=0)
id_experimento = np.concatenate(
    id_experimento_lista,
    axis=0
)

NOMES_CARACTERISTICAS = [
    'acc_x', 'acc_y', 'acc_z',
    'gyro_x', 'gyro_y', 'gyro_z'
]

print('Formato de X:', X.shape)
print('Formato de y:', y.shape)
print('Formato de id_experimento:', id_experimento.shape)
print('Características:', NOMES_CARACTERISTICAS)

O resultado esperado é:

```text
Formato de X: (39884, 6)
Formato de y: (39884,)
Formato de id_experimento: (39884,)
```

Nesse momento, `X` ainda é bidimensional: `(amostras, características)`. A terceira dimensão será criada posteriormente, quando dividirmos os sinais em janelas para a RNN.

## 9. Inspeção das primeiras amostras

In [ ]:
dados_usuario1 = pd.DataFrame(
    X,
    columns=NOMES_CARACTERISTICAS
)

dados_usuario1['atividade'] = y
dados_usuario1['experimento'] = id_experimento

dados_usuario1.head()

O `DataFrame` é útil para inspeção e apresentação. Para as operações numéricas e para a futura entrada da RNN, continuaremos usando principalmente as matrizes NumPy `X` e `y`.

## 10. Verificação das classes presentes

In [ ]:
classes, quantidades = np.unique(y, return_counts=True)

tabela_classes = pd.DataFrame({
    'classe': classes,
    'quantidade_amostras': quantidades,
    'duracao_segundos': quantidades / 50
})

tabela_classes

A classe 0 não é uma atividade da base. Ela identifica as amostras que não pertencem a nenhum intervalo registrado em `labels.txt`. No próximo notebook, investigaremos visualmente essas regiões e a distribuição das atividades.

## 11. Verificações de consistência

As instruções `assert` interrompem a execução caso alguma condição esperada não seja satisfeita.

In [ ]:
assert X.ndim == 2, 'X deve possuir duas dimensões nesta etapa.'
assert X.shape[1] == 6, 'X deve possuir seis características.'
assert len(X) == len(y), 'X e y devem ter a mesma quantidade de amostras.'
assert len(X) == len(id_experimento), (
    'Cada amostra deve possuir um identificador de experimento.'
)
assert not np.isnan(X).any(), 'X contém valores ausentes.'
assert set(experimentos) == {1, 2}, (
    'Esperavam-se os experimentos 1 e 2 para o usuário 1.'
)

print('Todas as verificações foram concluídas com sucesso.')

## 12. Conclusão

Neste notebook:

- localizamos os arquivos no Google Drive;
- carregamos os experimentos 1 e 2 do usuário 1;
- combinamos acelerômetro e giroscópio em uma matriz com seis características;
- construímos um rótulo para cada amostra;
- preservamos a identificação dos experimentos;
- verificamos a consistência das matrizes.

No Notebook 2, vamos visualizar os sinais, as atividades e os intervalos sem rótulo.